In [116]:
import pandas as pd
import numpy as np

# ── Dataset ────────────────────────────────────────────────────────────────────
data = {
    "name":    ["Alice","Bob","Charlie","Diana","Eve","Frank","Grace","Henry","Ivy","Jack"],
    "age":     [28, 34, 22, 45, 31, 52, 27, 38, 29, 41],
    "dept":    ["Eng","Mktg","Eng","HR","Eng","Finance","Mktg","Eng","HR","Finance"],
    "salary":  [95000, 72000, 61000, 84000, 110000, 130000, 68000, 99000, 77000, 120000],
    "yrs_exp": [4, 8, 1, 12, 6, 22, 3, 10, 5, 15],
    "rating":  [4.2, 3.8, 3.5, 4.7, 4.9, 4.1, 3.9, 4.4, 4.0, 4.6],
    "remote":  [True, False, True, False, True, False, True, True, False, False],
}
df = pd.DataFrame(data)


# ── 1. INSPECT ─────────────────────────────────────────────────────────────────
df.head()               # first 5 rows
df.tail(3)              # last 3 rows
df.shape                # (10, 7)
df.columns.tolist()     # column names
df.dtypes               # dtype of each column
df.info()               # non-null counts + dtypes
df.describe()           # stats for numeric cols
df.describe(include="all")  # include object/bool cols too


# ── 2. SELECT & FILTER ─────────────────────────────────────────────────────────
df["salary"]                         # single column → Series
df[["name", "salary", "dept"]]       # multiple columns → DataFrame

df.iloc[0]                           # first row by position
df.iloc[2:5]                         # rows 2,3,4
df.iloc[:, 1:4]                      # all rows, columns 1-3
df.iloc[0, 3]                        # scalar: row 0, col 3

df.loc[0]                            # first row by label
df.loc[2:4, ["name","salary"]]       # label slice + specific cols

df[df["dept"] == "Eng"]                                  # single condition
df[(df["salary"] > 90000) & (df["dept"] == "Eng")]       # AND
df[(df["salary"] > 100000) | (df["remote"] == True)]     # OR
df[~(df["dept"] == "HR")]                                # NOT

df[df["dept"].isin(["Eng", "Finance"])]                  # isin
df[df["salary"].between(70000, 100000)]                  # between
df[df["name"].str.startswith("A")]                       # string filter


# ── 3. COLUMN OPERATIONS ───────────────────────────────────────────────────────
df["salary_k"]   = df["salary"] / 1000                   # arithmetic
df["total_comp"] = df["salary"] * 1.1                    # bonus col
df["senior"]     = df["yrs_exp"] > 8                     # boolean col
df["grade"]      = pd.cut(df["rating"],
                          bins=[0,4.0,4.5,5.0],
                          labels=["B","A","S"])           # bin into categories

df.rename(columns={"yrs_exp": "experience", "dept": "department"}, inplace=True)
df.rename(columns={"experience": "yrs_exp", "department": "dept"}, inplace=True)  # revert

df.drop(columns=["total_comp"], inplace=True)            # drop column
df.drop(index=[0], inplace=True)                         # drop row by index
df.reset_index(drop=True, inplace=True)                  # reset after drop


# ── 4. SORTING & RANKING ───────────────────────────────────────────────────────
df.sort_values("salary", ascending=False)
df.sort_values(["dept", "salary"], ascending=[True, False])  # multi-col sort
df.nlargest(3, "salary")
df.nsmallest(3, "rating")
df["salary_rank"] = df["salary"].rank(ascending=False)   # rank col


# ── 5. GROUPBY ─────────────────────────────────────────────────────────────────
df.groupby("dept")["salary"].mean()
df.groupby("dept")["salary"].agg(["mean","min","max","count"])

df.groupby("dept").agg(
    avg_salary = ("salary", "mean"),
    headcount  = ("name",   "count"),
    avg_rating = ("rating", "mean"),
    max_exp    = ("yrs_exp","max"),
)

# transform: broadcast group stat back to original shape
df["dept_avg_salary"] = df.groupby("dept")["salary"].transform("mean")
df["salary_vs_dept"]  = df["salary"] - df["dept_avg_salary"]  # deviation


# ── 6. MISSING DATA ────────────────────────────────────────────────────────────
# Introduce NaNs
df_na = df.copy()
df_na.loc[1, "salary"] = np.nan
df_na.loc[3, "rating"] = np.nan
df_na.loc[5, "age"]    = np.nan

df_na.isna().sum()                        # nulls per column
df_na.isna().mean() * 100                 # % missing

df_na["salary"].fillna(df_na["salary"].mean(), inplace=True)  # fill with mean
df_na["rating"].fillna(method="ffill", inplace=True)          # forward fill
df_na.dropna(inplace=True)                                    # drop remaining
df_na.dropna(subset=["age"])                                  # drop only if age null
df_na.dropna(thresh=5)                                        # keep rows with ≥5 non-null


# ── 7. STRING OPERATIONS (.str) ────────────────────────────────────────────────
df["name"].str.upper()
df["name"].str.lower()
df["name"].str.len()
df["name"].str.contains("a", case=False)
df["name"].str.replace("a", "@", case=False)
df["dept"].str.startswith("E")
df["name"].str.split("").str[0]           # first character
df["dept"].value_counts()                 # frequency table


# ── 8. APPLY & MAP ─────────────────────────────────────────────────────────────
df["salary_band"] = df["salary"].apply(
    lambda x: "High" if x > 100000 else ("Mid" if x > 75000 else "Low")
)

df["name_dept"] = df.apply(lambda row: f"{row['name']} ({row['dept']})", axis=1)

mapping = {"Eng": "Engineering", "Mktg": "Marketing", "HR": "Human Resources", "Finance": "Finance"}
df["dept_full"] = df["dept"].map(mapping)   # map dict over series

df["rating_sq"] = df["rating"].map(lambda x: round(x**2, 2))


# ── 9. TRANSFORM & RESHAPE ─────────────────────────────────────────────────────
df["age"]  = df["age"].astype(float)
df["dept"] = df["dept"].astype("category")   # memory-efficient

# Normalize salary to 0-1
df["salary_norm"] = (df["salary"] - df["salary"].min()) / \
                    (df["salary"].max() - df["salary"].min())

# Z-score
df["salary_z"] = (df["salary"] - df["salary"].mean()) / df["salary"].std()

df["salary"].clip(lower=70000, upper=120000)   # cap outliers

# Cumulative ops
df["cumulative_salary"] = df["salary"].cumsum()
df["salary"].pct_change()                      # % change row-over-row


# ── 10. MERGE & JOIN ───────────────────────────────────────────────────────────
dept_info = pd.DataFrame({
    "dept":   ["Eng", "Mktg", "HR", "Finance"],
    "budget": [500000, 200000, 150000, 350000],
    "floor":  [3, 2, 4, 5],
})

pd.merge(df, dept_info, on="dept", how="left")    # left join (keep all df rows)
pd.merge(df, dept_info, on="dept", how="inner")   # inner join
pd.merge(df, dept_info, on="dept", how="outer")   # outer join

# Concat: stack rows
df_a = df.iloc[:5].copy()
df_b = df.iloc[5:].copy()
pd.concat([df_a, df_b], axis=0, ignore_index=True)

# Concat: add columns
pd.concat([df[["name","salary"]], df[["rating","remote"]]], axis=1)


# ── 11. PIVOT TABLE ────────────────────────────────────────────────────────────
df.pivot_table(
    values="salary",
    index="dept",
    aggfunc=["mean", "count", "max"]
)

df.pivot_table(
    values="salary",
    index="dept",
    columns="remote",    # split by remote True/False
    aggfunc="mean"
)

# Correlation matrix
df[["age", "salary", "yrs_exp", "rating"]].corr()


# ── 12. USEFUL UTILITIES ───────────────────────────────────────────────────────
df.duplicated().sum()          # count duplicate rows
df.drop_duplicates(inplace=True)

df.sample(5)                   # random 5 rows
df.sample(frac=0.3)            # random 30%

df["dept"].unique()            # unique values
df["dept"].nunique()           # number of unique values

df.set_index("name")           # use name as index
df.reset_index()               # revert back

df.copy()                      # explicit copy (avoids SettingWithCopyWarning)

df.to_csv("employees.csv", index=False)         # save
pd.read_csv("employees.csv")                    # load
df.to_excel("employees.xlsx", index=False)      # excel export

4